# Lesson 3.5 — RAG Pipeline: Chunking, Indexing, and Retrieval

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Design a chunking strategy suited to the document shape
- Explain indexing and the role of embeddings in retrieval
- Match a retrieval strategy to the data shape and query pattern

**You will produce:** A working `chunk_document()` plus a retrieval-strategy selector.

## Concepts

**RAG (Retrieval-Augmented Generation)** grounds Claude in your data: you **chunk** documents into passages, **embed** them (turn text into vectors that capture meaning), **index** them in a vector store, then at query time retrieve the most similar chunks and put them in the prompt.

**Chunking** = splitting text into retrievable units. Too large wastes context and buries the answer; too small loses context. Strategy depends on *document shape*: fixed-size+overlap for prose, structure-aware for Markdown/HTML, row/record for tables, function/class for code.

**Retrieval strategy** depends on *query pattern*: dense (semantic) for paraphrased questions, keyword/BM25 for exact terms/IDs, **hybrid** for both, and add **re-ranking** for precision. Sample 3 is a RAG failure: confident-but-wrong answers after a document refresh point to the retrieval/indexing step (stale/mismatched chunks), not the model.

> **Reference:** [Anthropic — Retrieval-augmented generation & embeddings](https://docs.anthropic.com/en/docs/build-with-claude/embeddings)

In [ ]:
# !pip install anthropic
import os, time, json

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic(); LIVE = True
    else:
        client = None; LIVE = False
except Exception as e:
    client = None; LIVE = False
    print("anthropic SDK unavailable:", e)

MODELS = {"opus":"claude-opus-4-20250514","sonnet":"claude-sonnet-4-20250514","haiku":"claude-3-5-haiku-20241022"}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    if not LIVE:
        return f"[MOCK {model}] would answer: {prompt[:80]}..."
    msg = client.messages.create(model=model, max_tokens=max_tokens,
        system=system or "You are a helpful assistant.",
        messages=[{"role":"user","content":prompt}])
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Implement fixed-size chunking with overlap

Overlap preserves context across boundaries so an answer split between two chunks is still retrievable. We chunk by approximate tokens (chars/4 heuristic).

In [ ]:
def chunk_document(text, chunk_tokens=200, overlap_tokens=40):
    words = text.split()
    # approx: 1 token ~ 0.75 words -> words per chunk
    wpc = int(chunk_tokens * 0.75); wov = int(overlap_tokens * 0.75)
    chunks, i = [], 0
    step = max(1, wpc - wov)
    while i < len(words):
        chunks.append(" ".join(words[i:i+wpc]))
        i += step
    return chunks

doc = ("Coverage policy. MRI requires prior authorization after a failed six week "
       "course of physical therapy. Emergency imaging is exempt. ") * 10
chunks = chunk_document(doc, 60, 15)
print(f"{len(chunks)} chunks; first chunk:\n{chunks[0][:160]}...")

## Code-Along: Step 2 — Choose chunking by document shape

One size does not fit all. A selector maps document type to the right chunking approach — an architectural decision reviewers will probe.

In [ ]:
def chunk_strategy(doc_type):
    return {
      "prose":      "fixed-size + overlap (semantic continuity)",
      "markdown":   "structure-aware (split on headings, keep sections intact)",
      "table":      "row/record-based (one row or record per chunk)",
      "code":       "syntax-aware (function/class boundaries)",
      "transcript": "speaker/turn or time-window based",
    }.get(doc_type, "fixed-size + overlap (default)")

for d in ["prose","markdown","table","code","transcript"]:
    print(f"{d:10} -> {chunk_strategy(d)}")

## Code-Along: Step 3 — Select a retrieval strategy by query pattern

Dense vs keyword vs hybrid depends on how users query. We encode the rule and note when to add re-ranking for precision.

In [ ]:
def retrieval_strategy(exact_terms_matter, semantic_paraphrase, precision_critical):
    if exact_terms_matter and semantic_paraphrase:
        base = "hybrid (BM25 + dense embeddings)"
    elif exact_terms_matter:
        base = "keyword/BM25 (IDs, codes, exact phrases)"
    else:
        base = "dense/semantic (embeddings)"
    return base + (" + re-ranking" if precision_critical else "")

print(retrieval_strategy(True, True, True))    # policy codes + NL questions, high stakes
print(retrieval_strategy(False, True, False))  # FAQ paraphrases
print(retrieval_strategy(True, False, False))  # lookup by claim ID

## Exercise

Diagnose Sample 3: a RAG system returns confident-but-wrong answers after a document refresh; model and latency unchanged. Write a `diagnose_rag()` that, given signals (embedding model changed?, index rebuilt?, retrieval_miss rate up?, chunks stale?), returns the most likely root cause — and confirm it points at the retrieval/indexing step, not the model.

In [ ]:
# TODO: implement diagnose_rag(signals) -> most likely cause
# signals example: {'reindexed': True, 'embedding_model_changed': True, 'retrieval_miss_up': True}
# TODO: show it returns a retrieval/indexing cause for the Sample 3 situation

## Key Takeaways

- RAG = chunk -> embed -> index -> retrieve -> ground the prompt.
- Match chunking to document shape; match retrieval to query pattern.
- Hybrid retrieval + re-ranking wins when both exact terms and meaning matter.
- Confident-but-wrong after a doc refresh = retrieval/indexing fault (Sample 3 = B).

---

**Next:** [Lesson 3.6 — Connection Protocols and Progressive Discovery](./lesson-3.6-connection-protocols.ipynb)